# Write Bürkle 2025 OD600 data as linked ATST

One study contains five source OD600 blocks: main Figures 1A and 6A, and supplementary Figures S1B, S1C and S6B. `Not done` becomes a missing reading, while the curve remains.

In [5]:
from pathlib import Path
import datetime
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    ReadoutIds,
    Study,
)

root = Path.cwd()
workbook = load_workbook(
    root / "sources/raw_data_set_wraf065.xlsx", read_only=True, data_only=True
)
# Each group: source first column, number of source experiment columns, entity ID, normalized treatment.
blocks = {
    "Fig_1A": (
        "Figure 1",
        5,
        7,
        12,
        "h",
        "PAO1",
        [
            (10, 3, "", "untreated"),
            (13, 3, "JG005", "single JG005"),
            (16, 3, "JG024", "single JG024"),
            (19, 3, "JG005_JG024", "simultaneous JG005+JG024"),
        ],
    ),
    "Fig_6A": (
        "Figure 6",
        4,
        6,
        11,
        "h",
        "PAO1",
        [
            (10, 3, "", "untreated"),
            (13, 3, "JG005_JG024", "simultaneous JG005+JG024"),
            (16, 3, "JG005_JG024", "sequential JG024 then JG005"),
        ],
    ),
    "Fig_S1B": (
        "Figure S1",
        82,
        84,
        88,
        "h",
        "PAO1_del_retS",
        [
            (10, 3, "", "untreated"),
            (13, 3, "JG005", "single JG005"),
            (16, 3, "JG024", "single JG024"),
            (19, 3, "JG005_JG024", "JG005+JG024 combination"),
        ],
    ),
    "Fig_S1C": (
        "Figure S1",
        92,
        94,
        99,
        "h",
        "CHA",
        [
            (10, 2, "", "untreated"),
            (12, 2, "JG005", "JG005"),
            (14, 2, "JG024", "JG024"),
            (16, 2, "JG005_JG024", "JG005+JG024 combination"),
        ],
    ),
    "Fig_S6B": (
        "Figure S6",
        7,
        9,
        153,
        "min",
        "PAO1",
        [
            (10, 3, "", "untreated"),
            (13, 3, "Bhz17", "Bhz17"),
            (16, 3, "JG005_JG024", "JG005+JG024"),
            (19, 3, "JG005_JG024_Bhz17", "JG005+JG024+Bhz17"),
        ],
    ),
}

In [6]:
file_info = FileMetadata(
    data={
        "file_name": "atst_burkle_2025.atst.txt",
        "format": "ATST",
        "format_version": "0.1",
        "created_on": datetime.date.today().isoformat(),
        "field_delimiter": "TAB",
        "encoding": "UTF-8",
    }
)
study = Study(
    data={
        "title": "Phage-phage competition and biofilms affect interactions between two virulent bacteriophages and Pseudomonas aeruginosa",
        "study_id": "doi:10.1093/ismejo/wraf065",
        "authors": "Magdalena Bürkle et al.",
        "publication_date": "2025-04-06",
        "publication_doi": "10.1093/ismejo/wraf065",
        "description": "Five reported OD600 blocks from planktonic phage assays across main and supplementary figures.",
    }
)
entities = Entities(
    tables={
        name: EntityTable(
            name="ENTITIES",
            table_name=name,
            pk=pk,
            data=pd.read_csv(
                root / "entities_definitions" / f"{name}.csv",
                dtype=str,
                keep_default_na=False,
            ),
        )
        for name, pk in [
            ("ISOLATES", "ISOLATE_ID"),
            ("PHAGES", "PHAGE_ID"),
            ("MEDIA", "MEDIUM_ID"),
        ]
    }
)

## Source measurements and six-column layouts

The panel rows and source-column groups are explicit above. Composite PHAGE_ID values link to local combination records whose components point to the individual phage rows. Source condition labels, including the Figure 6 `JG025` typo, remain in metadata and curation notes.

In [7]:
def curve_id(index):
    letters = ""
    while index:
        index, remainder = divmod(index - 1, 26)
        letters = chr(65 + remainder) + letters
    return "curve_" + letters


readouts = {}
for readout_id, (
    sheet_name,
    header_row,
    first_row,
    last_row,
    time_unit,
    isolate_id,
    groups,
) in blocks.items():
    sheet = workbook[sheet_name]
    times = [sheet.cell(row, 9).value for row in range(first_row, last_row + 1)]
    assert all(isinstance(t, (int, float)) for t in times)
    readings = pd.DataFrame({"Time": times})
    layout_rows = []
    source_groups = []
    missing = []
    for start, count, phage_id, treatment in groups:
        source_label = str(sheet.cell(header_row, start).value).strip()
        source_groups.append(
            f"{get_column_letter(start)}:{get_column_letter(start + count - 1)}={source_label}"
        )
        for replicate in range(1, count + 1):
            column = start + replicate - 1
            cid = curve_id(len(layout_rows) + 1)
            values = []
            for row in range(first_row, last_row + 1):
                value = sheet.cell(row, column).value
                if value == "Not done":
                    missing.append(sheet.cell(row, column).coordinate)
                    value = None
                assert value is None or isinstance(value, (int, float)), (
                    sheet_name,
                    sheet.cell(row, column).coordinate,
                    value,
                )
                values.append(value)
            readings[cid] = values
            layout_rows.append(
                {
                    "curve_id": cid,
                    "type": "uninfected_control" if not phage_id else "phage_exposed",
                    "isolate_id": isolate_id,
                    "phage_id": phage_id,
                    "treatment": treatment,
                    "replicate": str(replicate),
                }
            )
    layout = pd.DataFrame(layout_rows)
    context = {
        "Fig_1A": "PAO1 planktonic; paper Fig. 1A reports 1:1 phage-to-bacterium ratio; three biological replicates",
        "Fig_6A": "PAO1 planktonic; sequential JG024 precedes JG005 by 2 h at 1e7 PFU/mL each (paper Methods and Fig. 6)",
        "Fig_S1B": "PAO1 delta-retS planktonic; supplementary figure workbook; dose not verified in local paper",
        "Fig_S1C": "CHA planktonic; supplementary figure workbook; dose not verified in local paper",
        "Fig_S6B": "PAO1 growth with Bhz17 and phage combinations; dense 10-min worksheet series",
    }[readout_id]
    assay = Assay(
        data={
            "readout_type": "absorbance",
            "readout_unit": "od600",
            "time_unit": time_unit,
            "medium_id": "LB",
            "culture_type": "planktonic",
            "conditions": context,
            "temperature_c": "37",
            "atmosphere": "5% CO2",
            "starting_od600": "approximately 0.01 reported for general planktonic assay",
            "source_reference": "Paper Methods, P. aeruginosa-phage interaction; paper figure captions; source workbook panel heading",
        }
    )
    metadata = Metadata(
        data={
            "source_file": "read_shared_data/sources/raw_data_set_wraf065.xlsx",
            "source_time_range": f"I{first_row}:I{last_row}",
            "source_data_rows": f"{first_row}:{last_row}",
            "source_column_groups": " | ".join(source_groups),
            "source_missing_cells": ", ".join(missing),
            "conversion": "Numeric source values and source times preserved; source 'Not done' represented by empty READINGS cells; group labels normalized in LAYOUT",
        }
    )
    readouts[readout_id] = ATSTFile(
        file_info=file_info,
        study=study,
        metadata=metadata,
        assay=assay,
        entities=entities,
        layout=Layout(data=layout),
        readings=Readings(data=readings),
    )
readout_ids = ReadoutIds(data=pd.DataFrame({"readout_id": list(readouts)}))
for item in readouts.values():
    item.readout_ids = readout_ids
bundle = MultiReadoutATST(file_info, study, readout_ids, readouts, entities)
validate_atst(bundle)
pd.DataFrame(
    [
        {
            "readout_id": k,
            "curves": len(v.layout.data),
            "time_points": len(v.readings.data),
        }
        for k, v in readouts.items()
    ]
)

,readout_id,curves,time_points
0,Fig_1A,12,6
1,Fig_6A,9,6
2,Fig_S1B,12,5
3,Fig_S1C,8,6
4,Fig_S6B,12,145


In [4]:
write_multi_readout_atst(
    bundle, root.parent / "atst_burkle_2025", linked_files=True
)